# E-commerce Sales Data Analysis

**Portfolio Project — Python, Pandas, NumPy, Matplotlib & Seaborn**

### Objective
Analyze raw e-commerce sales data, clean data quality issues, calculate business KPIs, visualize trends, and extract actionable business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "ecommerce_sales_raw.csv"
OUTPUT_DIR = BASE_DIR / "visualizations"
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

## 1. Understand the Raw Data

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)
print("\nMissing values:")
display(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 2. Data Cleaning

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Unit Price"] = pd.to_numeric(df["Unit Price"], errors="coerce")

before = len(df)
df = df.drop_duplicates().copy()

df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan
df.loc[df["Unit Price"] <= 0, "Unit Price"] = np.nan

df["City"] = df["City"].fillna("Unknown")
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Customer ID"] = df["Customer ID"].fillna("Unknown")

df = df.dropna(subset=["Order Date", "Quantity", "Unit Price"]).copy()

print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning:  {len(df)}")
print(f"Rows removed:        {before - len(df)}")

## 3. Feature Engineering

In [ ]:
df["Revenue"] = df["Quantity"] * df["Unit Price"]
df["Month"] = df["Order Date"].dt.to_period("M").astype(str)
df["Year"] = df["Order Date"].dt.year

df.head()

## 4. Key Business Metrics

In [ ]:
total_revenue = df["Revenue"].sum()
total_orders = df["Order ID"].nunique()
total_quantity = df["Quantity"].sum()
avg_order_value = df.groupby("Order ID")["Revenue"].sum().mean()
unique_customers = df["Customer ID"].nunique()

metrics = pd.Series({
    "Total Revenue": total_revenue,
    "Total Orders": total_orders,
    "Total Quantity Sold": total_quantity,
    "Average Order Value": avg_order_value,
    "Unique Customers": unique_customers
})
display(metrics)

## 5. Monthly Revenue

In [ ]:
monthly = df.groupby("Month", as_index=False)["Revenue"].sum()

plt.figure(figsize=(10,5))
plt.plot(monthly["Month"], monthly["Revenue"], marker="o")
plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "monthly_revenue.png", dpi=150)
plt.show()

## 6. Revenue by Category

In [ ]:
category = df.groupby("Category", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(data=category, x="Revenue", y="Category")
plt.title("Revenue by Category")
plt.xlabel("Revenue (₹)")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "revenue_by_category.png", dpi=150)
plt.show()

## 7. Revenue by Region

In [ ]:
region = df.groupby("Region", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=False)

plt.figure(figsize=(9,5))
sns.barplot(data=region, x="Revenue", y="Region")
plt.title("Revenue by Region")
plt.xlabel("Revenue (₹)")
plt.ylabel("Region")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "revenue_by_region.png", dpi=150)
plt.show()

## 8. Top 10 Products

In [ ]:
product = df.groupby("Product", as_index=False)["Revenue"].sum().sort_values("Revenue", ascending=False).head(10)

plt.figure(figsize=(10,6))
sns.barplot(data=product, x="Revenue", y="Product")
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue (₹)")
plt.ylabel("Product")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top_10_products.png", dpi=150)
plt.show()

## 9. Payment Methods

In [ ]:
payment = df.groupby("Payment Method", as_index=False)["Order ID"].nunique().sort_values("Order ID", ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(data=payment, x="Order ID", y="Payment Method")
plt.title("Orders by Payment Method")
plt.xlabel("Number of Orders")
plt.ylabel("Payment Method")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "orders_by_payment_method.png", dpi=150)
plt.show()

## 10. Business Insights

Use the results above to write 4–6 concise observations. Example categories:

- Which product category generated the most revenue?
- Which region performed best?
- Which month had the highest revenue?
- Which products are the strongest revenue contributors?
- Which payment method is most frequently used?

**Important:** Replace generic statements with the actual values produced by your analysis.

In [ ]:
print("Top category:", category.iloc[0]["Category"])
print("Top region:", region.iloc[0]["Region"])
print("Top product:", product.iloc[0]["Product"])
print("Most-used payment method:", payment.iloc[0]["Payment Method"])

## 11. Export Cleaned Data

The cleaned dataset can be delivered to a client as a separate CSV file.

In [ ]:
clean_path = BASE_DIR / "data" / "ecommerce_sales_cleaned.csv"
df.to_csv(clean_path, index=False)
print(f"Saved: {clean_path}")